# Brent Crude — Geopolitical Event Impact Analysis
**Middle East Crisis 2023–2026** | EDA & Event Study

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.stats as stats

ROOT    = Path().resolve().parent
RAW     = ROOT / 'raw'
PROC    = ROOT / 'processed'

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)

## 1. Load data

In [ ]:
brent = pd.read_csv(RAW / 'brent_prices.csv', index_col='date', parse_dates=True)
brent = brent[['open','high','low','close','volume']].dropna(subset=['close'])

events = pd.read_csv(RAW / 'geopolitical_events.csv', parse_dates=['date']).set_index('date').sort_index()

macro_files = {'dxy': 'macro_dxy.csv', 'vix': 'macro_vix.csv',
               'ttf': 'macro_ttf.csv', 'gold': 'macro_gold.csv', 'spx': 'macro_spx.csv'}
macro = pd.concat([
    pd.read_csv(RAW / f, index_col='date', parse_dates=True).rename(columns={col.upper(): name, 'Close': name})
    for name, f in macro_files.items()
], axis=1)
macro.index = pd.to_datetime(macro.index)

df = brent.join(macro, how='left').ffill()
df['ret']        = df['close'].pct_change()
df['log_ret']    = np.log(df['close'] / df['close'].shift(1))
df['vol_30d']    = df['ret'].rolling(30).std() * np.sqrt(252)
df['vol_7d']     = df['ret'].rolling(7).std()  * np.sqrt(252)

print(f'{len(df)} trading days  |  {df.index[0].date()} → {df.index[-1].date()}')
print(f'Current price : ${df.close.iloc[-1]:.2f}/bbl')
df.tail(3)

## 2. Price series & realized volatility

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.65, 0.35],
                    subplot_titles=['Brent Crude (USD/bbl)', 'Realized Volatility (annualized)'])

# Price
fig.add_trace(go.Scatter(x=df.index, y=df.close, name='Brent',
                         line=dict(color='#E8871E', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df.close.rolling(30).mean(), name='MA30',
                         line=dict(color='white', width=1, dash='dash')), row=1, col=1)

# Event markers — severity >= 4 only
major = events[events.severity >= 4]
for dt, row in major.iterrows():
    idx = df.index.get_indexer([dt], method='nearest')[0]
    price = df.close.iloc[idx]
    color = 'red' if str(row.get('strait_hormuz_risk','')) in ['high','very_high','critical'] else 'orange'
    fig.add_trace(go.Scatter(
        x=[dt], y=[price], mode='markers',
        marker=dict(size=9, color=color, symbol='triangle-up'),
        name=row['event'][:40], showlegend=False,
        hovertext=f"{row['event']}<br>Severity {row['severity']}/5"
    ), row=1, col=1)

# Vol
fig.add_trace(go.Scatter(x=df.index, y=df.vol_30d, name='Vol 30d',
                         line=dict(color='#4FC3F7', width=1.2),
                         fill='tozeroy', fillcolor='rgba(79,195,247,0.1)'), row=2, col=1)

fig.update_layout(template='plotly_dark', height=600,
                  title='Brent Crude 2020–2026 — Geopolitical Events Annotated',
                  hovermode='x unified', showlegend=False)
fig.show()

## 3. Return distribution

In [ ]:
rets = df.ret.dropna()

print('=== Daily returns descriptive stats ===')
print(rets.describe().to_string())
print(f'\nSkewness : {rets.skew():.3f}')
print(f'Kurtosis : {rets.kurtosis():.3f}  (normal = 0, fat tails > 0)')
print(f'\nTop 5 single-day drops:')
print(rets.nsmallest(5).to_string())
print(f'\nTop 5 single-day spikes:')
print(rets.nlargest(5).to_string())

In [ ]:
x = np.linspace(rets.min(), rets.max(), 300)
mu, sigma = rets.mean(), rets.std()

fig = go.Figure()
fig.add_trace(go.Histogram(x=rets, nbinsx=80, histnorm='probability density',
                           name='Observed', marker_color='#E8871E', opacity=0.7))
fig.add_trace(go.Scatter(x=x, y=stats.norm.pdf(x, mu, sigma),
                         name='Normal fit', line=dict(color='white', dash='dash')))
fig.update_layout(template='plotly_dark', height=380,
                  title='Daily Return Distribution — fat tails visible',
                  xaxis_title='Daily return', yaxis_title='Density')
fig.show()

## 4. Event impact study

For each major event (severity ≥ 4), measure cumulative returns at T+1, T+3, T+5, T+10, T+20, T+30.
This is a standard **event study** methodology used in finance research.

In [ ]:
WINDOWS = [1, 3, 5, 10, 20, 30]

def event_return(dt, df, window):
    """Cumulative return from event date to T+window trading days."""
    idx = df.index.get_indexer([dt], method='nearest')[0]
    if idx + window >= len(df):
        return np.nan
    p0 = df.close.iloc[idx]
    p1 = df.close.iloc[idx + window]
    return (p1 - p0) / p0

major = events[events.severity >= 4].copy()

for w in WINDOWS:
    major[f'ret_T+{w}'] = major.index.map(lambda dt: event_return(dt, df, w))

ret_cols = [f'ret_T+{w}' for w in WINDOWS]
display_cols = ['event', 'category', 'severity', 'strait_hormuz_risk'] + ret_cols

result = major[display_cols].copy()
for c in ret_cols:
    result[c] = result[c].map(lambda x: f'{x*100:+.1f}%' if pd.notna(x) else 'n/a')

print(f'{len(major)} major events (severity >= 4)\n')
result

In [ ]:
# Average cumulative return across all major events
avg = [major[f'ret_T+{w}'].mean() * 100 for w in WINDOWS]
med = [major[f'ret_T+{w}'].median() * 100 for w in WINDOWS]

fig = go.Figure()
fig.add_trace(go.Scatter(x=WINDOWS, y=avg, name='Mean',
                         mode='lines+markers', line=dict(color='#E8871E', width=2)))
fig.add_trace(go.Scatter(x=WINDOWS, y=med, name='Median',
                         mode='lines+markers', line=dict(color='#4FC3F7', width=2, dash='dash')))
fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(template='plotly_dark', height=380,
                  title='Average Brent Return After Major Geopolitical Event',
                  xaxis_title='Trading days after event', yaxis_title='Cumulative return (%)')
fig.show()

print('Average cumulative returns after major events:')
for w, a, m in zip(WINDOWS, avg, med):
    print(f'  T+{w:2d}d : mean={a:+.2f}%  median={m:+.2f}%')

In [ ]:
# Price paths ±30 days around each event (normalized to 100 at T=0)
PRE, POST = 10, 30

fig = go.Figure()
for dt, row in major.iterrows():
    idx = df.index.get_indexer([dt], method='nearest')[0]
    if idx < PRE or idx + POST >= len(df):
        continue
    window_prices = df.close.iloc[idx - PRE: idx + POST + 1].values
    normalized    = window_prices / window_prices[PRE] * 100
    x_axis        = list(range(-PRE, POST + 1))
    color = 'red' if str(row.get('strait_hormuz_risk','')).lower() in ['high','very_high','critical'] else '#E8871E'
    fig.add_trace(go.Scatter(
        x=x_axis, y=normalized, mode='lines',
        line=dict(color=color, width=1),
        opacity=0.45,
        name=row['event'][:35],
        hovertemplate='%{y:.1f}<extra>' + row['event'][:35] + '</extra>'
    ))

# Average path
paths = []
for dt, row in major.iterrows():
    idx = df.index.get_indexer([dt], method='nearest')[0]
    if idx < PRE or idx + POST >= len(df):
        continue
    w = df.close.iloc[idx - PRE: idx + POST + 1].values
    paths.append(w / w[PRE] * 100)

avg_path = np.nanmean(paths, axis=0)
fig.add_trace(go.Scatter(x=list(range(-PRE, POST+1)), y=avg_path,
                         mode='lines', name='Average',
                         line=dict(color='white', width=3)))

fig.add_vline(x=0, line_dash='dash', line_color='yellow', annotation_text='Event')
fig.add_hline(y=100, line_dash='dot', line_color='gray')
fig.update_layout(template='plotly_dark', height=450,
                  title='Brent Price Paths Around Major Geopolitical Events (T=0 → 100)',
                  xaxis_title='Trading days relative to event',
                  yaxis_title='Normalized price', showlegend=False)
fig.show()

## 5. Macro correlations

In [ ]:
# Correlation on log returns — more meaningful than levels
log_rets = pd.DataFrame({
    'brent': df.log_ret,
    'dxy':   np.log(df.dxy   / df.dxy.shift(1)),
    'vix':   np.log(df.vix   / df.vix.shift(1)),
    'ttf':   np.log(df.ttf   / df.ttf.shift(1)),
    'gold':  np.log(df.gold  / df.gold.shift(1)),
    'spx':   np.log(df.spx   / df.spx.shift(1)),
}).dropna()

corr = log_rets.corr()
print('Correlation matrix (log returns):')
print(corr.to_string())

In [ ]:
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, aspect='auto')
fig.update_layout(template='plotly_dark', height=420,
                  title='Correlation Matrix — Daily Log Returns')
fig.show()

In [ ]:
# Rolling 90d correlation: Brent vs DXY and TTF
roll_dxy = df.log_ret.rolling(90).corr(np.log(df.dxy / df.dxy.shift(1)))
roll_ttf = df.log_ret.rolling(90).corr(np.log(df.ttf / df.ttf.shift(1)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=roll_dxy, name='Brent / DXY',
                         line=dict(color='#E8871E', width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=roll_ttf, name='Brent / TTF',
                         line=dict(color='#4FC3F7', width=1.5)))
fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(template='plotly_dark', height=380,
                  title='Rolling 90d Correlation — Brent vs DXY & TTF',
                  yaxis_title='Pearson r', yaxis=dict(range=[-1, 1]))
fig.show()

print(f'\nFull-period correlations with Brent:')
print(corr['brent'].drop('brent').sort_values().to_string())

## 6. Volatility around events

In [ ]:
# Compare realized vol in event windows vs. normal periods
df['near_event'] = 0
for dt in major.index:
    idx = df.index.get_indexer([dt], method='nearest')[0]
    lo  = max(0, idx - 5)
    hi  = min(len(df), idx + 6)
    df.iloc[lo:hi, df.columns.get_loc('near_event')] = 1

vol_event  = df[df.near_event == 1]['vol_7d'].dropna()
vol_normal = df[df.near_event == 0]['vol_7d'].dropna()

print('Realized volatility (7d annualized):')
print(f'  During event windows : {vol_event.mean():.1%}  (n={len(vol_event)})')
print(f'  Normal periods       : {vol_normal.mean():.1%}  (n={len(vol_normal)})')
print(f'  Ratio                : {vol_event.mean()/vol_normal.mean():.2f}x')

t_stat, p_val = stats.ttest_ind(vol_event, vol_normal)
print(f'\nt-test: t={t_stat:.2f}, p={p_val:.4f} — difference is {"significant" if p_val < 0.05 else "not significant"} at 5%')

---
## Summary

Key findings to carry into the forecasting notebook:

In [ ]:
print('=== KEY FINDINGS ===')
print(f"1. Return distribution: kurtosis={rets.kurtosis():.1f} — heavy tails, non-normal")
print(f"   Largest single-day drop  : {rets.min()*100:.1f}% on {rets.idxmin().date()}")
print(f"   Largest single-day spike : {rets.max()*100:.1f}% on {rets.idxmax().date()}")
print(f"\n2. Event impact (avg across {len(paths)} major events):")
for w, a in zip(WINDOWS, avg):
    print(f"   T+{w:2d}d : {a:+.2f}%")
print(f"\n3. Correlation with Brent (log returns):")
print(corr['brent'].drop('brent').sort_values().to_string())
print(f"\n4. Volatility is {vol_event.mean()/vol_normal.mean():.2f}x higher in ±5d event windows")